# States and Effects

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter
from io import BytesIO

import torch
from IPython.display import Image, Markdown, SVG, display
from matplotlib import pyplot as plt

from examples.algorithms.bell_tensor_workflow import bell_program, contract_bell_program
from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler.registers.layout.int_vector import int_effect_bindings, int_state_ir, int_vector_cost_key
from flagquantum.compiler.registers.primitives.basis_state import basis_effect_cost_key, basis_state_cost_key, basis_state_ir
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.register_types import opaque_register_type, quantum_bit_type
from flagquantum.core.ir.program.registers import Register, RegisterSide
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg
from flagquantum.drawer.register_graph import export_register_graph_dot
from flagquantum.drawer.register_score import RegisterStep, layout_register_score
from flagquantum.simulation.tensor_network.entrypoints import build_tensor_network, tensor_network_basis_effect

dtype = torch.complex128

def show_signature(name, signature):
    display(SVG(field_signature_svg([(name, signature, {})])))

def show_graph(name, key, *, integer=False):
    root = name if integer else key
    calls = {root: {key: 1}} if integer else {}
    graph = symbolic_call_graph(root, calls, max_depth=1)
    def label(node):
        return "ArbitraryClifford(8)" if integer and node == key else name
    display(SVG(render_call_graph_svg(export_call_graph_dot(root, graph.direct_calls, title=label))))
    display(Markdown(format_call_graph_markdown(root, graph.direct_calls, graph.leaf_counts, title=label)))

## `ZeroState`
The state |0>

In [ ]:
zero_state_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.RIGHT),))

### Example Instances

In [ ]:
zero_state = basis_state_cost_key("z", 0)

#### Graphical Signature

In [ ]:
show_signature("ZeroState", zero_state_signature)

### Call Graph

In [ ]:
show_graph("ZeroState", zero_state)

## `ZeroEffect`
The effect <0|

In [ ]:
zero_effect_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.LEFT),))

### Example Instances

In [ ]:
zero_effect = basis_effect_cost_key("z", 0)

#### Graphical Signature

In [ ]:
show_signature("ZeroEffect", zero_effect_signature)

### Call Graph

In [ ]:
show_graph("ZeroEffect", zero_effect)

## `OneState`
The state |1>

In [ ]:
one_state_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.RIGHT),))

### Example Instances

In [ ]:
one_state = basis_state_cost_key("z", 1)

#### Graphical Signature

In [ ]:
show_signature("OneState", one_state_signature)

### Call Graph

In [ ]:
show_graph("OneState", one_state)

## `OneEffect`
The effect <1|

In [ ]:
one_effect_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.LEFT),))

### Example Instances

In [ ]:
one_effect = basis_effect_cost_key("z", 1)

#### Graphical Signature

In [ ]:
show_signature("OneEffect", one_effect_signature)

### Call Graph

In [ ]:
show_graph("OneEffect", one_effect)

## `IntState`
The state |val> for non-negative integer val

#### Parameters
 - `val`: the classical value
 - `bitsize`: The bitsize of the register 

#### Registers
 - `val`: The register of size `bitsize` which initializes the value `val`.


In [ ]:
int_state_signature = Signature((Register("val", opaque_register_type(8), side=RegisterSide.RIGHT),))

### Example Instances

In [ ]:
int_state = int_vector_cost_key(8)

#### Graphical Signature

In [ ]:
show_signature("IntState", int_state_signature)

### Call Graph

In [ ]:
show_graph("IntState", int_state, integer=True)

## `IntEffect`
The effect <val| for non-negative integer val

#### Parameters
 - `val`: the classical value
 - `bitsize`: The bitsize of the register 

#### Registers
 - `val`: The register of size `bitsize` which de-allocates the value `val`.


In [ ]:
int_effect_signature = Signature((Register("val", opaque_register_type(8), side=RegisterSide.LEFT),))

### Example Instances

In [ ]:
int_effect = int_vector_cost_key(8)

#### Graphical Signature

In [ ]:
show_signature("IntEffect", int_effect_signature)

### Call Graph

In [ ]:
show_graph("IntEffect", int_effect, integer=True)

## `PlusState`
The state |+>

In [ ]:
plus_state_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.RIGHT),))

### Example Instances

In [ ]:
plus_state = basis_state_cost_key("x", 0)

#### Graphical Signature

In [ ]:
show_signature("PlusState", plus_state_signature)

### Call Graph

In [ ]:
show_graph("PlusState", plus_state)

## `PlusEffect`
The effect <+|

In [ ]:
plus_effect_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.LEFT),))

### Example Instances

In [ ]:
plus_effect = basis_effect_cost_key("x", 0)

#### Graphical Signature

In [ ]:
show_signature("PlusEffect", plus_effect_signature)

### Call Graph

In [ ]:
show_graph("PlusEffect", plus_effect)

## `MinusState`
The state |->

In [ ]:
minus_state_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.RIGHT),))

### Example Instances

In [ ]:
minus_state = basis_state_cost_key("x", 1)

#### Graphical Signature

In [ ]:
show_signature("MinusState", minus_state_signature)

### Call Graph

In [ ]:
show_graph("MinusState", minus_state)

## `MinusEffect`
The effect <-|

In [ ]:
minus_effect_signature = Signature((Register("q", quantum_bit_type(), side=RegisterSide.LEFT),))

### Example Instances

In [ ]:
minus_effect = basis_effect_cost_key("x", 1)

#### Graphical Signature

In [ ]:
show_signature("MinusEffect", minus_effect_signature)

### Call Graph

In [ ]:
show_graph("MinusEffect", minus_effect)

## Properties and Relations

### Creating a Bell state

We can create a bell state ($|00\rangle + |11\rangle$) by cnotting a $|+\rangle$ state into a $|0\rangle$ state.

In [ ]:
ports = ("q0", "q1")
layout = layout_register_score(dict.fromkeys(ports, 1), (), (
    RegisterStep("PlusState", outputs=("q0",)),
    RegisterStep("ZeroState", outputs=("q1",)),
    RegisterStep("CNOT", ports, ports, "cx"),
))
display(SVG(render_call_graph_svg(export_register_graph_dot(layout))))
bell = contract_bell_program(bell_program(prepared=True), torch.empty((), dtype=dtype))
print(bell)

The physical Bell circuit prepares the two output wires with H and CX. The diagram uses the existing FlagQuantum circuit drawer.

In [ ]:
instructions = (*basis_state_ir("x", 0, 0), *basis_state_ir("z", 0, 1), Instruction("cx", (0, 1)))
circuit = Circuit(2, dtype=dtype)
circuit.append(FieldOperation(2, instructions))
torch.testing.assert_close(circuit.state().flatten(), bell.flatten(), rtol=0, atol=2e-12)
fig, _ = circuit.draw("mpl")
with BytesIO() as png:
    fig.savefig(png, format="png", bbox_inches="tight")
    display(Image(data=png.getvalue()))
plt.close(fig)

### Prepare and contract exact vectors

Prepare each basis ket using the native append path. Apply the corresponding effect to a complex probe without normalization. The eight-bit example prepares and selects integer 55 in most-significant-bit order. The integer graph models each state or effect as one Clifford leaf, while the prepared ket emits one X for every set bit.

In [ ]:
probe = torch.tensor([1 + 2j, -3 + 4j], dtype=dtype)
ket_vectors = (("z", 0, [1, 0]), ("z", 1, [0, 1]), ("x", 0, [2**-0.5, 2**-0.5]), ("x", 1, [2**-0.5, -2**-0.5]))
for basis, bit, values in ket_vectors:
    ket = torch.tensor(values, dtype=dtype)
    prep = Circuit(1, dtype=dtype)
    prep.append(FieldOperation(1, basis_state_ir(basis, bit, 0)))
    torch.testing.assert_close(prep.state().flatten(), ket, atol=2e-12, rtol=0)
    plan = build_tensor_network(Circuit(1, inputs=probe, dtype=dtype), dtype=dtype)
    actual = tensor_network_basis_effect(plan, {0: (basis, bit)}).reshape(())
    torch.testing.assert_close(actual, torch.vdot(ket, probe), atol=2e-12, rtol=0)

wires = tuple(range(8))
int_instructions = int_state_ir(wires, 55)
integer_circuit = Circuit(8, dtype=dtype).append(FieldOperation(8, int_instructions))
expected = torch.zeros(256, dtype=dtype)
expected[55] = 1
torch.testing.assert_close(integer_circuit.state().flatten(), expected, atol=0, rtol=0)
plan = build_tensor_network(integer_circuit, dtype=dtype)
overlap = tensor_network_basis_effect(plan, int_effect_bindings(wires, 55)).reshape(())
torch.testing.assert_close(overlap, torch.ones((), dtype=dtype), atol=0, rtol=0)
print("IntState emitted", dict(Counter(g.name for g in int_instructions)))
print("Integer state and effect model", query_qec_gate_cost(int_state))